# Phase B (Notebook 1): Build & Analyze Spoiler-Aware 3-Tier GraphRAG Index (`PhaseB1_GraphRAG_Indexing.ipynb`)

**Environment Stack**: Runs on a **free Google Colab CPU runtime** (0 GPU compute units required). Uses `DeepSeek-V3` (`deepseek-chat`) for Tier-2 progressive chapter synthesis & entity/relationship extraction, and local CPU dense embeddings (`BAAI/bge-base-en-v1.5`, 768-dim via `sentence-transformers`).

### Workflow Overview
1. **Cell 1 — Google Drive Mount, `DEEPSEEK_API_KEY` & Phase-A Bundle Check**:
   - Mounts Google Drive, loads `DEEPSEEK_API_KEY` from **Colab Secrets** (key icon on left sidebar) or manual override, and verifies `movie_knowledge_bundle.json` and `keyframes/` inside `/content/drive/MyDrive/MovieRAG/workdir/{MOVIE_ID}`.
2. **Cell 2 — Clone / Pull Latest Code from GitHub (`/content/MovieRag_repo`)**:
   - Clones/pulls `main` and registers `MovieRAG/videorag` on `sys.path`.
3. **Cell 3 — Install Lightweight CPU Dependencies**:
   - Installs `openai>=1.50.0`, `httpx`, `tenacity`, `tiktoken`, `networkx`, `nano-vectordb`, `sentence-transformers`, `pillow`, and `nest_asyncio`.
4. **Cell 4 — Inspect Input `movie_knowledge_bundle.json`**:
   - Prints scene counts, total duration, character glossary entries, external plot documents, and previews merged micro-beat scene captions.
5. **Cell 5 — Build the 3-Tier GraphRAG Index (`rag.ainsert_knowledge_bundle()`)**:
   - Builds **Tier 1 (`scene`)** chunks (`1 Scene = 1 Rich Chunk`), **Tier 2 (`chapter`)** progressive summaries (every `CHAPTER_WINDOW_SEC`), and **Tier 3 (`full_movie`)** plot chunks.
   - Extracts time-sliced entities and relationships (`first_seen_time` + `temporal_descriptions`) into `graph_chunk_entity_relation.graphml`, `vdb_chunks.json`, and `vdb_entities.json`.
6. **Cell 6 — Analyze the 3-Tier Chunks (`scene`, `chapter`, `full_movie`)**:
   - Displays token distributions by tier and prints the generated Tier-2 Progressive Chapter Summaries.
7. **Cell 7 — Analyze the Time-Sliced Knowledge Graph & `vdb_entities.json`**:
   - Inspects extracted character nodes, `first_seen_time` timestamps, chronological `temporal_descriptions` slices, and 1-hop relationship edges.

In [ ]:
# ==============================================================================
# CELL 1: Google Drive Mount, DeepSeek API Key & Phase-A Bundle Verification
# ==============================================================================
import os

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running inside Google Colab; using local paths.")

MOVIE_ID = "sintel"                              # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/MovieRAG"   # @param {type:"string"}
API_BASE_URL = "https://api.deepseek.com"        # @param {type:"string"}
EMBEDDING_PROVIDER = "sentence_transformers"     # @param ["sentence_transformers", "auto", "siliconflow", "ollama", "hash"]
ST_EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"     # @param {type:"string"}
DEEPSEEK_API_KEY_OVERRIDE = ""                   # @param {type:"string"}

WORKDIR = f"{DRIVE_ROOT}/workdir/{MOVIE_ID}"
BUNDLE_PATH = f"{WORKDIR}/movie_knowledge_bundle.json"

os.environ["DEEPSEEK_BASE_URL"] = API_BASE_URL
os.environ["MOVIERAG_EMBEDDING_PROVIDER"] = EMBEDDING_PROVIDER
os.environ["MOVIERAG_ST_MODEL"] = ST_EMBEDDING_MODEL
os.environ["MOVIERAG_EMBEDDING_DIM"] = "768"

_api_key = DEEPSEEK_API_KEY_OVERRIDE.strip() or os.environ.get("DEEPSEEK_API_KEY", "").strip()
if not _api_key and IN_COLAB:
    try:
        _api_key = (userdata.get("DEEPSEEK_API_KEY") or "").strip()
    except Exception:
        _api_key = ""

if _api_key:
    os.environ["DEEPSEEK_API_KEY"] = _api_key
    print(f"[OK] DEEPSEEK_API_KEY loaded (starts with '{_api_key[:6]}...')")
else:
    print("[WARNING] DEEPSEEK_API_KEY not found! Add 'DEEPSEEK_API_KEY' in Colab Secrets (key icon on left sidebar) or paste it into DEEPSEEK_API_KEY_OVERRIDE.")

for label, path in [
    ("movie_knowledge_bundle.json", BUNDLE_PATH),
    ("keyframes dir", f"{WORKDIR}/keyframes"),
]:
    ok = os.path.exists(path)
    print(f"  [{'OK' if ok else 'MISSING'}] {label}: {path}")

In [ ]:
# ==============================================================================
# CELL 2: Clone / Pull Latest Source Code from GitHub (/content/MovieRag_repo)
# ==============================================================================
import importlib
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/ctu-flix/MovieRag.git"  # @param {type:"string"}
GITHUB_BRANCH = "main"                                        # @param {type:"string"}
CLONE_DIR = "/content/MovieRag_repo"

github_token = None
if globals().get("IN_COLAB", False):
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

auth_repo_url = GITHUB_REPO_URL
if github_token and GITHUB_REPO_URL.startswith("https://github.com/"):
    auth_repo_url = GITHUB_REPO_URL.replace("https://github.com/", f"https://{github_token}@github.com/")

if not os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"Cloning {GITHUB_REPO_URL} (branch: {GITHUB_BRANCH})...")
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, auth_repo_url, CLONE_DIR], check=True)
else:
    print(f"Pulling latest commits from origin/{GITHUB_BRANCH}...")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", auth_repo_url], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)

PROJECT_ROOT = CLONE_DIR if os.path.isdir(os.path.join(CLONE_DIR, "MovieRAG")) else os.path.join(CLONE_DIR, "MovieRag")
MOVIERAG_PKG_DIR = os.path.join(PROJECT_ROOT, "MovieRAG")

for p in (MOVIERAG_PKG_DIR, PROJECT_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

for mod_name in [k for k in list(sys.modules) if k == "videorag" or k.startswith("videorag.")]:
    del sys.modules[mod_name]

commit_log = subprocess.run(["git", "-C", CLONE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(f"Active PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"Active MOVIERAG_PKG   : {MOVIERAG_PKG_DIR}")
print(f"Latest Commit         : {commit_log}")

In [ ]:
# ==============================================================================
# CELL 3: Install Lightweight CPU GraphRAG & Embedding Dependencies
# ==============================================================================
import subprocess
import sys

pkgs = [
    "openai>=1.50.0",
    "httpx",
    "tenacity",
    "tiktoken",
    "networkx",
    "nano-vectordb",
    "sentence-transformers",
    "pillow",
    "nest_asyncio",
]
print("Installing Phase B dependencies:", ", ".join(pkgs))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed"] + pkgs,
    check=True,
)

import nest_asyncio
nest_asyncio.apply()

import videorag
print("Successfully loaded videorag package:", videorag.__file__)

In [ ]:
# ==============================================================================
# CELL 4: Inspect Input movie_knowledge_bundle.json Before Indexing
# ==============================================================================
import json
import os

with open(BUNDLE_PATH, "r", encoding="utf-8") as f:
    bundle = json.load(f)

scenes = bundle.get("scenes", [])
glossary = bundle.get("character_glossary", {})
ext_docs = bundle.get("external_plot_documents", [])
total_subs = sum(len(s.get("subtitles", [])) for s in scenes)
total_kfs = sum(len(s.get("keyframes", [])) for s in scenes)

print(f"Movie ID               : {bundle.get('movie_id')}")
print(f"Movie Title            : {bundle.get('movie_title')}")
print(f"Duration               : {bundle.get('duration_sec')}s ({bundle.get('duration_sec', 0)/60:.1f} min)")
print(f"Total Narrative Scenes : {len(scenes)}")
print(f"Total Subtitles        : {total_subs}")
print(f"Total Keyframes        : {total_kfs}")
print(f"Character Glossary     : {list(glossary.keys())}")
print(f"External Plot Docs     : {len(ext_docs)} (Tier 3 'full_movie')")

print("\n--- Sample Scene Preview (First 2 Scenes) ---")
for sc in scenes[:2]:
    chars = [c.get("name") if isinstance(c, dict) else str(c) for c in sc.get("characters_present", [])]
    print(f"\n[{sc['scene_id']}] {sc['start_time']:.2f}s - {sc['end_time']:.2f}s | Characters: {chars}")
    print((sc.get("vlm_caption") or "")[:400] + ("..." if len(sc.get("vlm_caption", "")) > 400 else ""))

In [ ]:
# ==============================================================================
# CELL 5: Build the 3-Tier Spoiler-Aware GraphRAG Index
# ==============================================================================
import asyncio
import os
import shutil
import time
from videorag import VideoRAG, deepseek_config

CHAPTER_WINDOW_SEC = 300.0   # @param {type:"number"}
REBUILD_FROM_SCRATCH = True  # @param {type:"boolean"}
CLEAR_LLM_CACHE = False      # @param {type:"boolean"}

index_files = [
    "kv_store_video_segments.json",
    "kv_store_text_chunks.json",
    "vdb_chunks.json",
    "vdb_entities.json",
    "graph_chunk_entity_relation.graphml",
]
if CLEAR_LLM_CACHE:
    index_files.append("kv_store_llm_response_cache.json")

if REBUILD_FROM_SCRATCH:
    archive_dir = os.path.join(WORKDIR, "archive_prev_graphrag")
    os.makedirs(archive_dir, exist_ok=True)
    removed = []
    for fname in index_files:
        fpath = os.path.join(WORKDIR, fname)
        if os.path.isfile(fpath):
            shutil.copy2(fpath, os.path.join(archive_dir, fname))
            os.remove(fpath)
            removed.append(fname)
    if removed:
        print(f"Archived & cleared previous index files -> {removed}")

t0 = time.perf_counter()
rag = VideoRAG(
    working_dir=WORKDIR,
    llm=deepseek_config,
    chapter_window_sec=CHAPTER_WINDOW_SEC,
)

print(f"Starting 3-Tier GraphRAG indexing on {BUNDLE_PATH}...")
stats = asyncio.get_event_loop().run_until_complete(
    rag.ainsert_knowledge_bundle(BUNDLE_PATH)
)
elapsed = time.perf_counter() - t0

print(f"\n[DONE in {elapsed:.1f}s] Indexing Summary:")
for k, v in stats.items():
    print(f"  - {k:26s}: {v}")

print("\nSaved Index Artifacts in WORKDIR:")
for fname in index_files + ["kv_store_llm_response_cache.json"]:
    fpath = os.path.join(WORKDIR, fname)
    if os.path.isfile(fpath):
        kb = os.path.getsize(fpath) / 1024.0
        print(f"  [OK] {fname:38s} ({kb:7.1f} KB)")

In [ ]:
# ==============================================================================
# CELL 6: Analyze the 3-Tier Chunks (Tier 1: scene, Tier 2: chapter, Tier 3: full_movie)
# ==============================================================================
import json
import os
from collections import Counter

chunks_path = os.path.join(WORKDIR, "kv_store_text_chunks.json")
with open(chunks_path, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

tier_counts = Counter(c.get("tier", "unknown") for c in all_chunks.values())
print(f"Total Indexed Chunks: {len(all_chunks)}")
for tier in ("scene", "chapter", "full_movie"):
    t_chunks = [c for c in all_chunks.values() if c.get("tier") == tier]
    if t_chunks:
        toks = [c.get("tokens", 0) for c in t_chunks]
        print(
            f"  - Tier '{tier:10s}': {len(t_chunks):3d} chunks | "
            f"tokens min={min(toks)}, avg={sum(toks)/len(toks):.0f}, max={max(toks)}"
        )

chapter_chunks = sorted(
    [c for c in all_chunks.values() if c.get("tier") == "chapter"],
    key=lambda c: float(c.get("start_time", 0.0)),
)
print(f"\n--- Generated Tier-2 Progressive Chapter Summaries ({len(chapter_chunks)}) ---")
for ch in chapter_chunks:
    print(f"\n[{ch['start_time']:.1f}s - {ch['end_time']:.1f}s] Characters: {ch.get('characters', [])}")
    print(ch.get("content", ""))

In [ ]:
# ==============================================================================
# CELL 7: Analyze the Time-Sliced Knowledge Graph & vdb_entities.json
# ==============================================================================
import json
import os

graph = rag.chunk_entity_relation_graph._graph
vdb_ent_path = os.path.join(WORKDIR, "vdb_entities.json")
with open(vdb_ent_path, "r", encoding="utf-8") as f:
    vdb_ent_payload = json.load(f)

vdb_ent_rows = vdb_ent_payload.get("data", [])
print(f"Knowledge Graph Nodes          : {graph.number_of_nodes()}")
print(f"Knowledge Graph Edges          : {graph.number_of_edges()}")
print(f"Time-Sliced Entity VDB Vectors : {len(vdb_ent_rows)}")

# Sort nodes by degree descending
node_list = []
for n_id in graph.nodes:
    n_data = graph.nodes[n_id]
    deg = graph.degree(n_id)
    raw_slices = n_data.get("temporal_descriptions", "[]")
    try:
        slices = json.loads(raw_slices)
    except Exception:
        slices = []
    node_list.append((n_id, deg, float(n_data.get("first_seen_time", 0.0)), n_data.get("entity_type", ""), slices))

node_list.sort(key=lambda x: (-x[1], x[2]))

print("\n--- Top 12 Knowledge Graph Entities (Time-Sliced) ---")
for n_id, deg, fst, etype, slices in node_list[:12]:
    print(f"\nEntity [{n_id}] (Type: {etype} | Degree: {deg} | First Seen: {fst:.1f}s | Slices: {len(slices)})")
    for s in slices[:4]:
        print(f"  • [{s.get('start_time', 0):6.1f}s - {s.get('end_time', 0):6.1f}s | {s.get('tier', 'scene'):10s}] {s.get('description', '')[:160]}")
    if len(slices) > 4:
        print(f"  • ... (+{len(slices) - 4} more temporal slices)")

print("\n--- Top 10 Time-Sliced Relationship Edges ---")
edge_items = list(graph.edges(data=True))
edge_items.sort(key=lambda e: -float(e[2].get("weight", 1.0)))
for u, v, edata in edge_items[:10]:
    efst = float(edata.get("first_seen_time", 0.0))
    w = float(edata.get("weight", 1.0))
    try:
        eslices = json.loads(edata.get("temporal_descriptions", "[]"))
    except Exception:
        eslices = []
    print(f"\nRelation [{u} <-> {v}] (Weight: {w:.1f} | First Seen: {efst:.1f}s | Slices: {len(eslices)})")
    for s in eslices[:3]:
        print(f"  • [{s.get('start_time', 0):6.1f}s - {s.get('end_time', 0):6.1f}s | {s.get('tier', 'scene'):10s}] {s.get('description', '')[:160]}")